[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/07_shell_tool.ipynb)

# 07. A Shell Tool

**Part 2: Tools** · about 25 minutes

An agent that can read and edit files but not use a terminal is like a developer who isn't allowed to run anything. With a shell it can run tests, search with `grep`, check `git status`, install packages and try a fix, all with one tool.

That same power is why the shell is the most dangerous tool an agent can have. This notebook builds the tool and its *mechanical* safety: timeouts that really stop things, output that can't flood the context, and an environment that doesn't hand over your secrets. Deciding *which* commands may run comes in notebook 08.

> **Running this with a real model:** the agent will run commands chosen by the model on your machine. Until notebook 08 adds proper permissions, this notebook allows only a short list of read-only commands. Colab is the safest place to experiment.

**By the end you will have:**

- watched a timeout fail to stop a command, and fixed it
- output clipping that keeps the part that matters
- a `run_command` tool, and an agent that uses it

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()

## 1. Running a command

Python's `subprocess` does the work. We use `shell=True` because the model writes *shell* syntax, like pipes, wildcards and `&&`, and only a shell understands that. Keep that sentence in mind, because it's also the reason for everything in notebook 08.

In [3]:
done = subprocess.run(
    "echo hello from the shell && ls /no/such/folder",
    shell=True, cwd=workspace, capture_output=True, text=True,
)

print("exit code:", done.returncode)
print("stdout:   ", done.stdout.strip())
print("stderr:   ", done.stderr.strip())

exit code: 1
stdout:    hello from the shell
stderr:    ls: /no/such/folder: No such file or directory


Three things come back: an **exit code** (0 means success, by convention), **stdout** and **stderr**. An agent needs all three. A failing command often explains itself only in stderr.

## 2. The timeout that doesn't stop anything

Agents start commands that never end, such as a dev server, a test waiting for input, or an accidental infinite loop. So we need a timeout. `subprocess.run` has one built in, and it looks like it works:

In [4]:
marker = workspace / "marker.txt"
command = "(sleep 2; echo still running > marker.txt); echo done"  # the ( ) starts a child process

started = time.perf_counter()
try:
    subprocess.run(command, shell=True, cwd=workspace, capture_output=True, text=True, timeout=1)
except subprocess.TimeoutExpired:
    print(f"Timed out after {time.perf_counter() - started:.1f}s, as expected.")

time.sleep(2.5)
print("Did the 'stopped' command keep running anyway?", marker.exists())

Timed out after 1.0s, as expected.


Did the 'stopped' command keep running anyway? True


**It kept running.** The timeout killed the shell, but not the child process the shell had started. That child finished its sleep and wrote the file anyway.

Now imagine the child is a test suite that writes to a database, a `git push`, or a server holding a port. The agent was told it stopped, but it didn't. These orphaned processes are a classic agent bug, and they're nasty to track down because nothing reports an error.

The fix is to start the command in its own **process group**, then kill the whole group.

In [5]:
def run_with_timeout(command, timeout, cwd, env=None):
    """Run a shell command, and on timeout kill it and everything it started."""
    options = dict(shell=True, cwd=cwd, env=env, text=True,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if os.name == "posix":
        options["start_new_session"] = True  # the shell and its children share a new process group

    process = subprocess.Popen(command, **options)
    try:
        output, _ = process.communicate(timeout=timeout)
        return process.returncode, output
    except subprocess.TimeoutExpired:
        if os.name == "posix":
            os.killpg(process.pid, signal.SIGKILL)  # the whole group, not just the shell
        else:
            subprocess.run(["taskkill", "/T", "/F", "/PID", str(process.pid)], capture_output=True)
        output, _ = process.communicate()
        return None, output


marker.unlink(missing_ok=True)
code, _ = run_with_timeout(command, timeout=1, cwd=workspace)
time.sleep(2.5)

print("Exit code:", code, "(None means we stopped it)")
print("Did it keep running this time?", marker.exists())

Exit code: None (None means we stopped it)
Did it keep running this time? False


In [6]:
# Checkpoint: after a timeout, nothing the command started may still be running.
assert code is None and not marker.exists()
print("Checkpoint passed.")

Checkpoint passed.


`stderr=subprocess.STDOUT` merges the two streams into one, in the order they happened. An agent reads output like a person reads a terminal, so the interleaving helps more than keeping the streams apart.

## 3. Output that can't flood the context

One careless `cat` of a log file can return megabytes. That's more than a context window can hold, and even when it fits, you pay for every token on every later step (remember the cost table in notebook 04).

The fix is to clip the output, but *where* you clip matters. Errors and summaries usually come at the **end**, so we keep the start and the end and cut the middle.

In [7]:
def clip(text, limit=4000):
    """Keep the beginning and the end, and say exactly how much was cut."""
    if len(text) <= limit:
        return text
    half = limit // 2
    return f"{text[:half]}\n[... {len(text) - limit:,} characters cut ...]\n{text[-half:]}"


code, output = run_with_timeout(
    f'{shlex.quote(sys.executable)} -c "for i in range(100000): print(i)"', timeout=30, cwd=workspace
)
clipped = clip(output, limit=60)

print(f"Raw output: {len(output):,} characters. After clipping: {len(clipped):,}.\n")
print(clipped)

Raw output: 588,890 characters. After clipping: 94.

0
1
2
3
4
5
6
7
8
9
10
11
12
1
[... 588,830 characters cut ...]
99995
99996
99997
99998
99999



The marker line matters. It tells the model that output was cut and by how much, so it doesn't assume it saw everything. A model that knows will often rerun with `grep` or `tail` to find what it needs.

## 4. Don't hand over your secrets

A child process inherits every environment variable of its parent by default, and that includes API keys. Anything the agent runs, including test files *the model wrote*, could read them.

In [8]:
os.environ["DEMO_SECRET"] = "pretend-this-is-your-api-key"

_, inherited = run_with_timeout("echo secret: $DEMO_SECRET", timeout=5, cwd=workspace)


def minimal_env():
    """Only what commands need to work. HOME points at the workspace so tools keep their files in there."""
    env = {name: os.environ[name] for name in ("PATH", "SYSTEMROOT", "LANG") if name in os.environ}
    env["HOME"] = str(workspace)
    return env


_, scrubbed = run_with_timeout("echo secret: $DEMO_SECRET", timeout=5, cwd=workspace, env=minimal_env())
del os.environ["DEMO_SECRET"]

print("Default environment: ", inherited.strip())
print("Minimal environment: ", scrubbed.strip())

Default environment:  secret: pretend-this-is-your-api-key
Minimal environment:  secret:


## 5. The tool

Here it all comes together. One more detail is `use_our_python`. The model will type `python`, but that name points at different interpreters on different machines, and on some it doesn't exist at all. We swap in the interpreter running this notebook, so commands behave the same everywhere.

In [9]:
PYTHON = shlex.quote(sys.executable)


def use_our_python(command):
    """Make `python` mean the interpreter running this notebook."""
    return re.sub(r"(?<![\w./-])python3?(?=\s|$)", lambda _: PYTHON, command)


def run_command(command, timeout=30):
    """Run a shell command in the workspace and describe what happened."""
    code, output = run_with_timeout(use_our_python(command), timeout, cwd=workspace, env=minimal_env())
    output = clip(output.replace(str(workspace), "."))
    status = f"stopped: took longer than {timeout} seconds" if code is None else f"exit code: {code}"
    return f"{status}\n{output}"


print(run_command("echo hi && python --version"))

exit code: 0
hi
Python 3.13.3



## 6. A temporary seatbelt

The agent is almost ready, but anything the model types would run with your user permissions. Notebook 08 builds real permissions. For now, a crude guard allows a few read-only commands and nothing that chains or redirects. It's deliberately blunt, and the next notebook explains why checks like this are harder than they look.

In [10]:
DEMO_ALLOWED = ("grep ", "ls", "cat ", "wc ", "head ", "tail ", "python -m unittest")
SHELL_OPERATORS = (";", "&", "|", ">", "<", "$(", "`", "\n")


def demo_guard(command):
    if any(operator in command for operator in SHELL_OPERATORS):
        return "Denied: this notebook doesn't allow chaining, pipes or redirects."
    if not command.strip().startswith(DEMO_ALLOWED):
        return f"Denied: this notebook only allows commands starting with {', '.join(DEMO_ALLOWED)}."
    return None


def guarded_run_command(command):
    return demo_guard(command) or run_command(command)


for attempt in ["ls", "rm -rf build", "ls; rm -rf build"]:
    print(f"{attempt!r:<22} -> {guarded_run_command(attempt).splitlines()[0]}")

'ls'                   -> exit code: 0
'rm -rf build'         -> Denied: this notebook only allows commands starting with grep , ls, cat , wc , head , tail , python -m unittest.
'ls; rm -rf build'     -> Denied: this notebook doesn't allow chaining, pipes or redirects.


## 7. The agent with a shell

Here's a small to-do app with a few `TODO` comments and some tests. We'll ask a question the agent can only answer by running commands.

In [11]:
(workspace / "app.py").write_text('''\
"""A tiny to-do app."""

TASKS = []


def add(task):
    # TODO: reject empty tasks
    TASKS.append(task)


def done(index):
    # TODO: handle an index that doesn't exist
    return TASKS.pop(index)


def summary():
    # TODO: say "1 task", not "1 tasks"
    return f"{len(TASKS)} tasks left"
''')

(workspace / "test_app.py").write_text('''\
import unittest

import app


class AppTests(unittest.TestCase):
    def setUp(self):
        app.TASKS.clear()

    def test_add_and_done(self):
        app.add("write notebook 08")
        self.assertEqual(app.done(0), "write notebook 08")

    def test_summary(self):
        app.add("a")
        app.add("b")
        self.assertEqual(app.summary(), "2 tasks left")


if __name__ == "__main__":
    unittest.main()
''')

marker.unlink(missing_ok=True)
print(sorted(p.name for p in workspace.iterdir()))

['app.py', 'test_app.py']


In [12]:
TOOLS = [{"type": "function", "function": {
    "name": "run_command",
    "description": ("Run a shell command in the project folder. Returns the exit code, then the output. "
                    "Long output is cut in the middle. Use `python` to run Python."),
    "parameters": {"type": "object",
                   "properties": {"command": {"type": "string", "description": "The command to run"}},
                   "required": ["command"]},
}}]

SYSTEM_PROMPT = "You are a coding agent with a terminal. Use commands to find answers, then reply briefly."


def call_model(messages):
    payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def run_agent(task, max_steps=10):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        message = call_model(messages)["choices"][0]["message"]
        messages.append(message)
        calls = message.get("tool_calls") or []
        if not calls:
            return {"status": "done", "answer": message.get("content")}
        for call in calls:
            try:
                command = json.loads(call["function"]["arguments"] or "{}").get("command", "")
                result = guarded_run_command(command)
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            print(f"[{step}] $ {command}")
            for line in result.splitlines()[:6]:
                print(f"    {line}")
    return {"status": "max_steps", "answer": None}


run = run_agent("How many TODO comments are in this project, and do the tests pass?")
print("\nAnswer:", run["answer"])

[1] $ grep -rn TODO --include=*.py .
    exit code: 0
    ./app.py:7:    # TODO: reject empty tasks
    ./app.py:12:    # TODO: handle an index that doesn't exist
    ./app.py:17:    # TODO: say "1 task", not "1 tasks"


[2] $ python -m unittest -q
    exit code: 0
    ----------------------------------------------------------------------
    Ran 2 tests in 0.000s
    
    OK



Answer: There are 3 TODO comments, and the tests pass.


In [13]:
# Checkpoint: the offline script counts the three TODOs and sees the tests pass.
if llm.offline:
    assert run["status"] == "done"
    assert "3 TODO" in run["answer"] and "pass" in run["answer"]
print("Checkpoint passed.")

Checkpoint passed.


Notice what the agent didn't need: no `count_todos` tool and no `run_tests` tool. One general tool replaced a whole set of specific ones. That's why every popular coding agent ships a shell, and also why notebook 08 exists.

## Recap

- The model writes shell syntax, so we need `shell=True`, which is exactly what makes the tool dangerous.
- A timeout has to kill the whole **process group**. Otherwise "stopped" commands keep running.
- Clip output in the middle, keep the end where errors live, and say how much was cut.
- Run commands with a minimal environment so your keys stay yours.

## Exercises

1. Real agents often keep one shell session open, so `cd` and `export` persist between commands. What would you have to change? What new risks does a persistent session bring?
2. Make `run_command` stream output as it arrives, so you can watch a long test run live.
3. Find a command that gets past `demo_guard` and does something it shouldn't. (There's at least one. Notebook 08 discusses it.)

## Clean up

In [14]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.


**Next:** [08. Permissions](08_permissions.ipynb). Deciding what the agent may run, and why prompts can't do that job.